# Курсова робота з дисципліни «Теорія керування. Частина 2»
**Тема:** побудова та дослідження прогнозуючого регулятора для керування портфелем активів

**Варіант (портфель) № 14.** Слюсар Олександр, група КА-33. Керівник: Канцедал Г. О.

Ноутбук містить розрахунки до всіх шести задач курсової роботи:
1. Період дискретизації за теоремою Котельникова.
2. Модель портфеля і прогнозуючий регулятор (аналітичний розв'язок і розв'язок з обмеженнями).
3. Вплив якості прогнозу на якість керування.
4. Прогнозування стандартними моделями (AR+РМНК, ARMA, ARIMA, Хольт) і керування за прогнозами.
5. Прогнозування нейронними мережами та ансамблем.
6. Модель авторегресії курсу BTC: передаточна функція, стійкість, керованість.

Дані завантажуються з Yahoo Finance і Binance при першому запуску й зберігаються в папці `data/`.
Рисунки зберігаються в `figs/`, підсумкові числа — в `results/`. Повний прогін займає близько 40 хв (2 ядра).

In [ ]:
import os, json, time, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import scipy.sparse as sp
from scipy import signal
from scipy.signal import detrend
from scipy.stats import jarque_bera, kurtosis, skew
from multiprocessing import Pool
import osqp, quadprog
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from IPython.display import display

warnings.filterwarnings("ignore")
os.makedirs("data", exist_ok=True); os.makedirs("figs", exist_ok=True); os.makedirs("results", exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.3, "figure.dpi": 110, "savefig.dpi": 170})
pd.set_option("display.width", 160)
SUMMARY = {}
NPROC = 2

PORTFOLIO = [
    ("METAB", "Штучний інтелект", "META"), ("MSFTB", "Штучний інтелект", "MSFT"),
    ("VALEB", "Гірнича здобич", "VALE"), ("USARB", "Гірнича здобич", "USAR"),
    ("FCXB", "Золото/Мідь", "FCX"), ("NEMB", "Золото/Мідь", "NEM"),
    ("ALABB", "Високі технології", "ALAB"), ("ASMLB", "Високі технології", "ASML"),
    ("BBB", "Програмне забезпечення", "BB"), ("ADBEB", "Програмне забезпечення", "ADBE"),
    ("MUB", "Фінанси", "MU"), ("JPMB", "Фінанси", "JPM"),
    ("XLEB", "Енергетика", "XLE"), ("FLNCB", "Енергетика", "FLNC"),
    ("ASTSB", "Аерокосмос", "ASTS"), ("RTXB", "Аерокосмос", "RTX"),
    ("PEPB", "Їжа", "PEP"), ("KOB", "Їжа", "KO"),
    ("AMZNB", "Медіа/Товари", "AMZN"), ("NFLXB", "Медіа/Товари", "NFLX"),
    ("BTC", "Криптовалюти", "BTC"), ("BNB", "Криптовалюти", "BNB"),
]
STOCKS = [p[2] for p in PORTFOLIO if p[1] != "Криптовалюти"]
START, END = "2025-04-01", "2026-10-16"
BINANCE = "https://data-api.binance.vision/api/v3"

## 0. Завантаження і підготовка даних
Токенізовані акції Binance (з «B» на кінці) є не для всіх тікерів варіанта, а частина з'явилась
лише кілька тижнів тому. Тому беремо ціни базових акцій з Yahoo Finance, а BTC і BNB — з Binance
(ціна о 16:00 за Нью-Йорком, тобто на момент закриття біржі NYSE). Нижче перевіряємо, що токени
справді повторюють ціну базових акцій.

In [ ]:
def get_json(url):
    for attempt in range(5):
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                return json.load(r)
        except Exception:
            time.sleep(2 + 2 * attempt)
    raise RuntimeError(url)

def binance_klines(symbol, interval, start, end):
    """Свічки Binance за період [start, end); індекс — час відкриття свічки (UTC)."""
    t0 = int(pd.Timestamp(start, tz="UTC").timestamp() * 1000)
    t1 = int(pd.Timestamp(end, tz="UTC").timestamp() * 1000)
    rows = []
    while t0 < t1:
        k = get_json(f"{BINANCE}/klines?symbol={symbol}&interval={interval}&startTime={t0}&endTime={t1 - 1}&limit=1000")
        if not k:
            break
        rows += k
        t0 = k[-1][0] + 1
        time.sleep(0.15)
    cols = ["open_time", "open", "high", "low", "close", "volume", "close_time",
            "quote_volume", "trades", "taker_buy_base", "taker_buy_quote", "ignore"]
    df = pd.DataFrame(rows, columns=cols).drop(columns="ignore")
    for c in ["open", "high", "low", "close", "volume", "quote_volume", "taker_buy_base", "taker_buy_quote"]:
        df[c] = df[c].astype(float)
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    return df.drop_duplicates("open_time").set_index("open_time")

def cached(path, loader):
    if not os.path.exists(path):
        loader().to_csv(path)
    return pd.read_csv(path, index_col=0, parse_dates=True)

def load_stocks():
    import yfinance as yf
    px = yf.download(STOCKS, start="2025-03-01", end="2026-10-17", interval="1d",
                     auto_adjust=True, progress=False, threads=False)["Close"][STOCKS]
    px.index = pd.to_datetime(px.index).tz_localize(None)
    return px

stocks_raw = cached("data/stocks_daily_raw.csv", load_stocks)
btc_h = cached("data/BTCUSDT_1h.csv", lambda: binance_klines("BTCUSDT", "1h", "2025-03-01", "2026-10-17"))
bnb_h = cached("data/BNBUSDT_1h.csv", lambda: binance_klines("BNBUSDT", "1h", "2025-03-01", "2026-10-17"))
btc_d = cached("data/BTCUSDT_1d.csv", lambda: binance_klines("BTCUSDT", "1d", "2022-01-01", "2026-10-17"))

if not os.path.exists("data/tokenized_status.json"):
    info = get_json(f"{BINANCE}/exchangeInfo?permissions=SPOT")
    have = {s["baseAsset"] for s in info["symbols"] if s["quoteAsset"] == "USDT" and s["status"] == "TRADING"}
    found = [v for v, _, _ in PORTFOLIO if v in have and v not in ("BTC", "BNB")]
    for v in found:
        binance_klines(v + "USDT", "1h", "2026-06-01", "2026-10-17").to_csv(f"data/tok_{v}_1h.csv")
    json.dump({"found": found, "missing": [v for v, _, _ in PORTFOLIO if v not in found and v not in ("BTC", "BNB")]},
              open("data/tokenized_status.json", "w"), indent=1)
tok_status = json.load(open("data/tokenized_status.json"))
print("знайдено на Binance:", tok_status["found"])
print("немає на Binance:  ", tok_status["missing"])

In [ ]:
def price_at_ny_close(h, dates):
    """Ціна закриття годинної свічки, що закінчується о 16:00 America/New_York."""
    end_t = (h.index + pd.Timedelta(hours=1)).tz_convert("America/New_York")
    s = pd.Series(h["close"].values, index=end_t)
    out = {}
    for d in dates:
        t = pd.Timestamp(d.date()).tz_localize("America/New_York") + pd.Timedelta(hours=16)
        out[d] = s.get(t, np.nan)
    return pd.Series(out)

px = stocks_raw.loc[START:END, STOCKS].copy()
px["BTC"] = price_at_ny_close(btc_h, px.index)
px["BNB"] = price_at_ny_close(bnb_h, px.index)
assert px.isna().sum().sum() == 0
px.to_csv("data/prices_22.csv")
names = list(px.columns)
P = px.values
XI = P[1:] / P[:-1] - 1
K, n = XI.shape
print("період:", px.index[0].date(), "-", px.index[-1].date(), "| днів:", len(px), "| кроків K =", K)

rows = []
for var, sector, base in PORTFOLIO:
    if var in ("BTC", "BNB"):
        rows.append(dict(sector=sector, var=var, used=f"{base}USDT", source="Binance", first="—", days=np.nan, corr=np.nan, dev=np.nan))
        continue
    if var not in tok_status["found"]:
        rows.append(dict(sector=sector, var=var, used=base, source="Yahoo Finance", first="немає", days=0, corr=np.nan, dev=np.nan))
        continue
    tok = price_at_ny_close(pd.read_csv(f"data/tok_{var}_1h.csv", index_col=0, parse_dates=True), px.index).dropna()
    a, b = tok, px.loc[tok.index, base]
    ra, rb = np.log(a).diff().dropna(), np.log(b).diff().dropna()
    rows.append(dict(sector=sector, var=var, used=base, source="Yahoo Finance", first=tok.index.min().strftime("%d.%m.%Y"),
                     days=len(tok), corr=ra.corr(rb) if len(ra) > 5 else np.nan, dev=float((a / b - 1).abs().mean() * 100)))
tok_tab = pd.DataFrame(rows)
tok_tab.to_csv("results/tab_portfolio.csv", index=False)
display(tok_tab)
ok = tok_tab["corr"].notna()
SUMMARY.update(tok_corr_min=tok_tab.loc[ok, "corr"].min(), tok_corr_max=tok_tab.loc[ok, "corr"].max(),
               tok_dev_min=tok_tab.loc[ok, "dev"].min(), tok_dev_max=tok_tab.loc[ok, "dev"].max(),
               N_days=len(px), K=K)

In [ ]:
# загальна динаміка та орієнтир «купив і тримай» (по 1/22 капіталу в кожен актив, комісія 0,05 %)
total_chg = (P[-1] / P[0] - 1) * 100
ew = (1e5 / (n * 1.0005) * P / P[0]).sum(1)
SUMMARY.update(mu_x=P[-1, names.index("MU")] / P[0, names.index("MU")],
               alab_x=P[-1, names.index("ALAB")] / P[0, names.index("ALAB")],
               adbe_pct=total_chg[names.index("ADBE")], nflx_pct=total_chg[names.index("NFLX")],
               pep_pct=total_chg[names.index("PEP")], btc_pct=total_chg[names.index("BTC")],
               bh_total=(ew[-1] / 1e5 - 1) * 100, bh_annual=((ew[-1] / 1e5) ** (252 / K) - 1) * 100,
               bh_dd=(ew / np.maximum.accumulate(ew) - 1).min() * 100,
               vol_min=XI.std(0).min() * 100, vol_max=XI.std(0).max() * 100,
               vol_min_asset=names[XI.std(0).argmin()], vol_max_asset=names[XI.std(0).argmax()])
print(pd.Series(total_chg, index=names).round(1).to_string())
print({k: round(v, 2) if isinstance(v, float) else v for k, v in SUMMARY.items()})

fig, axs = plt.subplots(4, 6, figsize=(12, 7.4), sharex=True)
for ax, c in zip(axs.ravel(), names):
    ax.plot(px.index, px[c] / px[c].iloc[0], lw=0.9, color="tab:blue")
    ax.axhline(1, color="gray", lw=0.6, ls="--")
    ax.set_title(c, fontsize=9, pad=2); ax.tick_params(labelsize=7)
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[4, 10]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%y"))
for ax in axs.ravel()[len(names):]:
    ax.axis("off")
for ax in list(axs[-1, :len(names) - 18]) + list(axs[-2, len(names) - 18:]):
    ax.xaxis.set_tick_params(labelbottom=True)
fig.tight_layout()
fig.savefig("figs/fig1_prices.png")
plt.show()

## Задача 1. Період дискретизації за теоремою Котельникова
Для кожного ряду: логарифм ціни мінус лінійний тренд → ДПФ (`numpy.fft.rfft`) → частка енергії
$S(f)$ → $f_{\max}$ — найменша частота, нижче якої лежить 99 % енергії → $T_{0,i}=1/(2f_{\max,i})$.
Спільний період $T_0=\min_i T_{0,i}$.

In [ ]:
def fmax_energy(price, thr=0.99, dt=1.0):
    x = detrend(np.log(np.asarray(price, float)), type="linear")
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(len(x), d=dt)
    E = np.abs(X[1:]) ** 2
    S = np.cumsum(E) / E.sum()
    return f[1:][np.searchsorted(S, thr)], f[1:], S

tab_t0 = []
for c in names:
    row = {"asset": c}
    for thr in (0.95, 0.99, 0.999):
        fm, _, _ = fmax_energy(px[c].values, thr)
        row[f"fmax_{thr}"] = fm
        row[f"T0_{thr}"] = 1 / (2 * fm)
    tab_t0.append(row)
tab_t0 = pd.DataFrame(tab_t0)
tab_t0.to_csv("results/tab_t0.csv", index=False)
display(tab_t0.round(3))
i99 = tab_t0["T0_0.99"].idxmin()
SUMMARY.update(T0_min=tab_t0.loc[i99, "T0_0.99"], fmax_min=tab_t0.loc[i99, "fmax_0.99"], T0_min_asset=tab_t0.loc[i99, "asset"],
               T0_min95=tab_t0["T0_0.95"].min(), T0_min95_asset=tab_t0.loc[tab_t0["T0_0.95"].idxmin(), "asset"],
               T0_min999=tab_t0["T0_0.999"].min(), T0_min999_asset=tab_t0.loc[tab_t0["T0_0.999"].idxmin(), "asset"])
T0 = int(np.floor(SUMMARY["T0_min"]))
print(f"min T0 = {SUMMARY['T0_min']:.3f} дня ({SUMMARY['T0_min_asset']}) -> беремо T0 = {T0} торговий день")

for nm, h in (("btc", btc_h), ("bnb", bnb_h)):
    hh = h.loc[(h.index >= pd.Timestamp(START, tz="UTC")) & (h.index < pd.Timestamp("2026-10-17", tz="UTC")), "close"]
    fm, _, _ = fmax_energy(hh.values, 0.99)
    SUMMARY[f"{nm}_h_fmax"] = fm; SUMMARY[f"{nm}_h_T0"] = 1 / (2 * fm)
SUMMARY["btc_h_T0d"] = SUMMARY["btc_h_T0"] / 24
print("BTC годинні: fmax = %.4f 1/год, T0 = %.1f год; BNB: T0 = %.1f год" % (SUMMARY["btc_h_fmax"], SUMMARY["btc_h_T0"], SUMMARY["bnb_h_T0"]))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.25]})
for c, col in (("MU", "tab:red"), ("BTC", "tab:orange"), ("BB", "tab:blue")):
    fm, f, S = fmax_energy(px[c].values, 0.99)
    a1.plot(f, S, color=col, label=f"{c}: $f_{{max}}$={fm:.3f}")
    a1.axvline(fm, color=col, ls=":", lw=1)
a1.axhline(0.99, color="k", lw=0.8, ls="--")
a1.set_xscale("log"); a1.set_xlabel("частота, циклів за торговий день"); a1.set_ylabel("частка енергії S(f)")
a1.set_ylim(0.6, 1.005); a1.legend(loc="lower right"); a1.set_title("а) накопичена енергія спектра")
t = tab_t0.sort_values("T0_0.99")
cols = ["tab:red" if a == SUMMARY["T0_min_asset"] else "tab:blue" for a in t["asset"]]
a2.bar(t["asset"], t["T0_0.99"], color=cols)
a2.axhline(1, color="k", ls="--", lw=0.9, label="вибраний крок $T_0$ = 1 день")
a2.set_ylabel("$T_{0,i}$, торгових днів"); a2.tick_params(axis="x", rotation=90, labelsize=8)
a2.legend(); a2.set_title("б) допустимий період дискретизації (поріг 99 %)")
fig.tight_layout(); fig.savefig("figs/fig2_spectrum.png"); plt.show()

## Задача 2. Модель портфеля і прогнозуючий регулятор
$x_i(k+1)=[1+\xi_i(k)][x_i(k)+p_i(k)-q_i(k)]$, $x_{n+1}(k+1)=(1-r)[x_{n+1}(k)-(1+\alpha)\sum p_i+(1-\beta)\sum q_i]$,
$y=\sum x_i$. Критерій $J=\sum_j Q e^2(k+j)+u^TRu$, задаюча траєкторія $y^*(k+j)=y(k)(1+g)^j$.
Оскільки $A(k)$, $B(k)$ змінюються з кроком, прогноз будується через добутки матриць (нестаціонарний випадок).

In [ ]:
class Portfolio:
    """x(k+1) = A(k) x(k) + B(k) u(k),  y = C x;  x = [x_1..x_n, гроші], u = [p_1..p_n, q_1..q_n]."""

    def __init__(self, n, alpha=5e-4, beta=5e-4, r=0.0):
        self.n, self.alpha, self.beta, self.r = n, alpha, beta, r
        self.nx, self.nu = n + 1, 2 * n
        self.C = np.ones((1, n + 1))

    def AB(self, xi):
        n, a, b, r = self.n, self.alpha, self.beta, self.r
        g = 1.0 + np.asarray(xi, float)
        A = np.diag(np.r_[g, 1.0 - r])
        B = np.zeros((n + 1, 2 * n))
        B[:n, :n] = np.diag(g)
        B[:n, n:] = -np.diag(g)
        B[n, :n] = -(1.0 - r) * (1.0 + a)
        B[n, n:] = (1.0 - r) * (1.0 - b)
        return A, B

    def step(self, x, u, xi):
        A, B = self.AB(xi)
        return A @ x + B @ u

    def prediction(self, Xi):
        """Phi, Gam такі, що x(k+j) = Phi_j x(k) + Gam_j u_hat, j = 1..Np."""
        Np, nx, nu = len(Xi), self.nx, self.nu
        Phi = np.zeros((Np * nx, nx)); Gam = np.zeros((Np * nx, Np * nu))
        P_prev, G_prev = np.eye(nx), np.zeros((nx, Np * nu))
        for j in range(Np):
            A, B = self.AB(Xi[j])
            d = np.diag(A)[:, None]
            P, G = d * P_prev, d * G_prev
            G[:, j * nu:(j + 1) * nu] = B
            Phi[j * nx:(j + 1) * nx], Gam[j * nx:(j + 1) * nx] = P, G
            P_prev, G_prev = P, G
        return Phi, Gam

class MPC:
    """Прогнозуючий регулятор з рухомим горизонтом."""

    def __init__(self, model, Np=5, Q=1.0, R=1e-5, g=0.0, constrained=True, solver="osqp", eps=1e-8, ref="relative"):
        self.m, self.Np, self.Q, self.R, self.g = model, Np, Q, R, g
        self.constrained, self.solver, self.eps, self.ref = constrained, solver, eps, ref
        self._osqp, self.k, self.y0 = None, 0, None

    def qp_matrices(self, xn, Xi, ycur):
        m, Np = self.m, len(Xi)
        Phi, Gam = m.prediction(Xi)
        Cbar = np.kron(np.eye(Np), m.C)
        Ahat, Bhat = Cbar @ Phi, Cbar @ Gam
        if self.ref == "relative":
            yref = (1 + self.g) ** np.arange(1, Np + 1)
        else:
            yref = self.y0 * (1 + self.g) ** (self.k + np.arange(1, Np + 1)) / ycur
        H = self.Q * Bhat.T @ Bhat + self.R * np.eye(Bhat.shape[1])
        f = self.Q * Bhat.T @ (Ahat @ xn - yref)
        return H, f, Phi, Gam

    def control(self, x, Xi):
        y = x.sum()
        if self.y0 is None:
            self.y0 = y
        xn = x / y
        Xi = np.asarray(Xi)[: self.Np]
        H, f, Phi, Gam = self.qp_matrices(xn, Xi, y)
        nU = H.shape[0]
        if not self.constrained:
            u = -np.linalg.solve(H, f)
        elif self.solver == "quadprog":
            u = quadprog.solve_qp(H, -f, np.vstack([np.eye(nU), Gam]).T, np.r_[np.zeros(nU), -Phi @ xn], 0)[0]
            u = np.maximum(u, 0)
        else:
            Pm = sp.triu(sp.csc_matrix(2 * H), format="csc")
            Am = sp.vstack([sp.eye(nU), sp.csc_matrix(Gam)]).tocsc()
            lo, hi = np.r_[np.zeros(nU), -Phi @ xn], np.full(Am.shape[0], np.inf)
            if self._osqp is None or (Pm.nnz, Am.nnz) != self._nnz:
                self._osqp = osqp.OSQP()
                self._osqp.setup(P=Pm, q=2 * f, A=Am, l=lo, u=hi, verbose=False, eps_abs=self.eps,
                                 eps_rel=self.eps, max_iter=40000, polish=True, warm_starting=True)
                self._nnz = (Pm.nnz, Am.nnz)
            else:
                self._osqp.update(Px=Pm.data, Ax=Am.data, q=2 * f, l=lo, u=hi)
            res = self._osqp.solve()
            u = np.maximum(res.x, 0) if res.x is not None else np.zeros(nU)
        self.k += 1
        return u[: self.m.nu] * y

def clip_feasible(model, x, u):
    """Захист від похибок розв'язувача на рівні 1e-9: не продаємо більше, ніж маємо, і не йдемо в мінус."""
    n = model.n
    p, q = np.maximum(u[:n], 0), np.maximum(u[n:], 0)
    q = np.minimum(q, x[:n] + p)
    cash = x[n] - (1 + model.alpha) * p.sum() + (1 - model.beta) * q.sum()
    if cash < 0 and p.sum() > 0:
        p = p * max(min((x[n] + (1 - model.beta) * q.sum()) / ((1 + model.alpha) * p.sum()), 1.0), 0.0)
    return np.r_[p, q]

def simulate(model, ctrl, XI, forecaster, x0):
    """Рішення приймаються за прогнозом forecaster(k), а стан змінюється за справжніми XI."""
    Kx, n = XI.shape
    X = np.zeros((Kx + 1, n + 1)); X[0] = x0
    U = np.zeros((Kx, 2 * n)); fees = 0.0
    for k in range(Kx):
        u = ctrl.control(X[k], forecaster(k))
        if ctrl.constrained:
            u = clip_feasible(model, X[k], u)
        U[k] = u
        fees += model.alpha * u[:n].sum() + model.beta * u[n:].sum()
        X[k + 1] = model.step(X[k], u, XI[k])
    return dict(X=X, U=U, y=X.sum(1), fees=fees)

def metrics(y, spy=252):
    y = np.asarray(y, float)
    return dict(final=y[-1], total=(y[-1] / y[0] - 1) * 100, annual=((y[-1] / y[0]) ** (spy / (len(y) - 1)) - 1) * 100,
                maxdd=(y / np.maximum.accumulate(y) - 1).min() * 100)

ALPHA = BETA = 5e-4
R_ANN, G_ANN = 0.05, 0.10
r_step, g_step = (1 + R_ANN) ** (1 / 252) - 1, (1 + G_ANN) ** (1 / 252) - 1
NP, RHO = 5, 1e-5
X0 = np.r_[np.zeros(n), 1e5]
model = Portfolio(n, ALPHA, BETA, r_step)
perfect = lambda k: XI[k:k + NP]
SUMMARY.update(r_step=r_step, g_step=g_step)
print(f"r = {r_step:.3e} за крок, g = {g_step:.3e} за крок")

In [ ]:
# Перевірка 1: матриці прогнозу (3.1) проти покрокового моделювання
rng = np.random.default_rng(0)
x_t, u_t = rng.random(n + 1), rng.random(NP * 2 * n) * 0.01
Phi, Gam = model.prediction(XI[:NP])
xs, err = x_t.copy(), 0.0
for j in range(NP):
    xs = model.step(xs, u_t[j * 2 * n:(j + 1) * 2 * n], XI[j])
    err = max(err, np.abs(xs - (Phi @ x_t + Gam @ u_t)[j * (n + 1):(j + 1) * (n + 1)]).max())
# Перевірка 2: керованість — ранг B(k) і матриці Калмана
ranks = [np.linalg.matrix_rank(model.AB(XI[k])[1]) for k in range(K)]
A_, B_ = model.AB(XI[0])
Kal = np.hstack([np.linalg.matrix_power(A_, i) @ B_ for i in range(n + 1)])
SUMMARY.update(pred_err=err, rankB_min=min(ranks), rankB_max=max(ranks), rank_kalman=np.linalg.matrix_rank(Kal))
print(f"похибка прогнозу стану: {err:.2e}; rank B(k): {min(ranks)}..{max(ranks)}; rank [B AB ...]: {SUMMARY['rank_kalman']}")

In [ ]:
# Аналітичний розв'язок і розв'язок з обмеженнями (OSQP) за ідеального прогнозу
t = time.time()
s_an = simulate(model, MPC(model, NP, 1.0, RHO, g_step, constrained=False), XI, perfect, X0)
t_an = time.time() - t
t = time.time()
s_qp = simulate(model, MPC(model, NP, 1.0, RHO, g_step, solver="osqp", eps=1e-8), XI, perfect, X0)
t_osqp = time.time() - t
t = time.time()
s_qp2 = simulate(model, MPC(model, NP, 1.0, RHO, g_step, solver="quadprog"), XI, perfect, X0)
t_qprog = time.time() - t
s_loose = simulate(model, MPC(model, NP, 1.0, RHO, g_step, solver="osqp", eps=1e-3), XI, perfect, X0)
m_an, m_qp, m_qp2, m_lo = (metrics(s["y"]) for s in (s_an, s_qp, s_qp2, s_loose))
inv_qp = (s_qp["X"][:, :n].sum(1) / s_qp["y"]).mean() * 100
SUMMARY.update(an_total=m_an["total"], an_annual=m_an["annual"], an_minpos=s_an["X"][:, :n].min(), an_fees=s_an["fees"],
               an_negshare=(s_an["U"] < 0).mean() * 100,
               qp_final=m_qp["final"], qp_total=m_qp["total"], qp_annual=m_qp["annual"], qp_dd=m_qp["maxdd"],
               qp_fees=s_qp["fees"], qp_inv=inv_qp, osqp_qp_diff=abs(m_qp["final"] - m_qp2["final"]),
               fees_tight=s_qp["fees"], fees_loose=s_loose["fees"], final_loose=m_lo["final"],
               t_osqp=t_osqp, t_qprog=t_qprog, speedup=t_osqp / t_qprog)
for nm, mm, s in (("аналітичний", m_an, s_an), ("OSQP 1e-8", m_qp, s_qp), ("quadprog", m_qp2, s_qp2), ("OSQP 1e-3", m_lo, s_loose)):
    print(f"{nm:12s}: капітал {mm['final']:.1f} $, {mm['annual']:.2f} % річних, просідання {mm['maxdd']:.3f} %, комісії {s['fees']:.1f} $")
print(f"мін. позиція (аналіт.) {SUMMARY['an_minpos']:.0f} $, від'ємних керувань {SUMMARY['an_negshare']:.0f} %; час OSQP {t_osqp:.1f} c, quadprog {t_qprog:.1f} c")

In [ ]:
def plot_run(s, title_prefix, fname, analytic=False):
    dates = px.index
    fig, axs = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
    axs[0].plot(dates, s["y"], color="tab:blue", label="капітал портфеля")
    axs[0].plot(dates, 1e5 * (1 + g_step) ** np.arange(K + 1), "k--", lw=0.8, label="план: 10 % річних")
    axs[0].axhline(1e5, color="tab:red", ls=":", lw=1, label="стартовий капітал")
    axs[0].set_ylabel("$"); axs[0].legend(); axs[0].set_title(f"{title_prefix}: загальний капітал")
    X = s["X"]
    if analytic:
        axs[1].plot(dates, np.where(X[:, :n] > 0, X[:, :n], 0).sum(1), label="сума довгих позицій")
        axs[1].plot(dates, np.where(X[:, :n] < 0, X[:, :n], 0).sum(1), label="сума коротких позицій")
        axs[1].plot(dates, X[:, n], label="гроші на рахунку")
        axs[1].set_title("розподіл капіталу")
    else:
        w = X[:, :n] / s["y"][:, None] * 100
        top = np.argsort(-w.mean(0))[:7]
        rest = np.delete(np.arange(n), top)
        axs[1].stackplot(dates, *[w[:, i] for i in top], w[:, rest].sum(1),
                         labels=[names[i] for i in top] + ["інші"], alpha=0.85)
        axs[1].set_ylabel("% капіталу"); axs[1].set_title("частка капіталу, вкладена в активи (решта — гроші на рахунку)")
    axs[1].legend(ncol=4, fontsize=8, loc="upper left")
    U = s["U"]
    if analytic:
        axs[2].plot(dates[:-1], U[:, :n].sum(1), color="tab:green", lw=0.9, label="купівлі Σp")
        axs[2].plot(dates[:-1], U[:, n:].sum(1), color="tab:red", lw=0.9, label="продажі Σq")
        axs[2].axhline(0, color="k", lw=0.6)
    else:
        axs[2].bar(dates[:-1], U[:, :n].sum(1), width=1.0, color="tab:green", label="купівлі Σp")
        axs[2].bar(dates[:-1], -U[:, n:].sum(1), width=1.0, color="tab:red", label="продажі −Σq")
    axs[2].set_ylabel("$ за день"); axs[2].legend(); axs[2].set_title("сигнали керування")
    axs[2].xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))
    fig.tight_layout(); fig.savefig(fname); plt.show()

plot_run(s_an, "Аналітичний розв'язок", "figs/fig3_analytic.png", analytic=True)
plot_run(s_qp, "Розв'язок з обмеженнями", "figs/fig4_qp.png")

In [ ]:
# Вибір ваги R і горизонту Np (ідеальний прогноз, g = 10 % річних)
def run_sweep(args):
    R, Np = args
    s = simulate(model, MPC(model, Np, 1.0, R, g_step, solver="quadprog"), XI, lambda k: XI[k:k + Np], X0)
    gr = np.diff(s["y"]) / s["y"][:-1]
    return dict(R=R, Np=Np, annual=metrics(s["y"])["annual"], track=np.sqrt(np.mean((gr - g_step) ** 2)) * 100,
                fees=s["fees"], inv=(s["X"][:, :n].sum(1) / s["y"]).mean() * 100)

jobs = [(R, 5) for R in (1e-3, 1e-4, 1e-5, 1e-6)] + [(1e-5, Np) for Np in (1, 3, 10, 20)]
with Pool(NPROC) as pool:
    tab_sweep = pd.DataFrame(pool.map(run_sweep, jobs, chunksize=1))
tab_sweep.to_csv("results/tab_sweep.csv", index=False)
display(tab_sweep.round(4))

In [ ]:
# Залежність від бажаного приросту g (ідеальний прогноз)
def run_g(args):
    g_ann, cons = args
    g = (1 + g_ann) ** (1 / 252) - 1
    s = simulate(model, MPC(model, NP, 1.0, RHO, g, constrained=cons, solver="quadprog"), XI, perfect, X0)
    mm = metrics(s["y"])
    return dict(g=g_ann * 100, cons=cons, annual=mm["annual"], total=mm["total"], maxdd=mm["maxdd"], fees=s["fees"],
                inv=(s["X"][:, :n].sum(1) / s["y"]).mean() * 100, minpos=s["X"][:, :n].min(), y=s["y"])

jobs = [(g, c) for g in (0.05, 0.10, 0.30, 1.0, 3.0, 10.0) for c in (True, False)]
with Pool(NPROC) as pool:
    res_g = pool.map(run_g, jobs, chunksize=1)
tab_g = pd.DataFrame([{k: v for k, v in r.items() if k != "y"} for r in res_g])
tab_g.to_csv("results/tab_gsweep.csv", index=False)
display(tab_g.round(2))

fig, ax = plt.subplots(figsize=(10, 4.6))
for r in res_g:
    if r["cons"] and r["g"] in (10, 30, 100):
        ax.plot(px.index, r["y"], label=f"MPC з обмеженнями, g = {r['g']:.0f} % річних")
ax.plot(px.index, ew, color="k", lw=1.2, label="«купив і тримай» (по 1/22)")
ax.axhline(1e5, color="gray", ls=":", lw=1)
ax.set_ylabel("капітал, $"); ax.legend(); ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))
fig.tight_layout(); fig.savefig("figs/fig5_gsweep.png"); plt.show()

## Задача 3. Вплив якості прогнозу на якість керування
Прогноз = справжня дохідність + шум: $\hat\xi_i=\xi_i+s_i(\mu+\lambda\varepsilon)$. Окремо: правильний знак
з випадковою величиною і точна величина з випадково зміненим знаком.

In [ ]:
S_I = XI.std(0)

def make_forecaster(kind, lam, mu, seed, flip):
    rng = np.random.default_rng(seed)
    E = rng.standard_normal((K, NP, n)); Fl = rng.random((K, NP, n)) < flip; Eta = np.abs(rng.standard_normal((K, NP, n)))

    def f(k):
        true = XI[k:k + NP]; L = len(true)
        if kind == "gauss":
            return true + (mu + lam * E[k][:L]) * S_I
        if kind == "sign":
            return np.sign(true) * Eta[k][:L] * S_I
        return np.where(Fl[k][:L], -true, true)
    return f

def run_noise(job):
    kind, lam, mu, flip, seed, g_ann = job
    f = make_forecaster(kind, lam, mu, seed, flip)
    s = simulate(model, MPC(model, NP, 1.0, RHO, (1 + g_ann) ** (1 / 252) - 1, solver="quadprog"), XI, f, X0)
    F1 = np.array([f(k)[0] for k in range(K)])
    return dict(kind=kind, lam=lam, mu=mu, flip=flip, seed=seed, g=g_ann * 100, annual=metrics(s["y"])["annual"],
                maxdd=metrics(s["y"])["maxdd"], sign=np.mean(np.sign(F1) == np.sign(XI)) * 100,
                corr=np.corrcoef(F1.ravel(), XI.ravel())[0, 1], inv=(s["X"][:, :n].sum(1) / s["y"]).mean() * 100, fees=s["fees"])

LAMS = (0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0)
jobs = [("gauss", lam, 0.0, 0.0, sd, g) for g in (0.1, 0.3, 1.0) for lam in LAMS for sd in range(3)]
jobs += [("gauss", 0.5, mu, 0.0, sd, 0.3) for mu in (-0.5, -0.25, 0.25, 0.5) for sd in range(3)]
jobs += [("sign", 0, 0, 0, sd, 0.3) for sd in range(3)]
jobs += [("flip", 0, 0, fl, sd, 0.3) for fl in (0.1, 0.2, 0.3, 0.4, 0.5) for sd in range(3)]
t = time.time()
with Pool(NPROC) as pool:
    noise = pd.DataFrame(pool.map(run_noise, jobs, chunksize=1))
print("час:", round(time.time() - t), "c")
noise.to_csv("results/noise_runs.csv", index=False)
nz = noise.groupby(["kind", "g", "lam", "mu", "flip"])[["annual", "maxdd", "sign", "corr", "inv", "fees"]].mean().reset_index()
nz.to_csv("results/tab_noise.csv", index=False)
display(nz.round(2))

In [ ]:
g_ = nz[nz.kind == "gauss"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))
for g, col in ((10, "tab:blue"), (30, "tab:orange"), (100, "tab:green")):
    d = g_[(g_.g == g) & (g_.mu == 0)]
    raw = noise[(noise.kind == "gauss") & (noise.g == g) & (noise.mu == 0)].groupby("lam")["annual"]
    a1.errorbar(d.lam, d.annual, yerr=[d.annual - raw.min().values, raw.max().values - d.annual], marker="o",
                color=col, capsize=3, label=f"g = {g} % річних")
a1.axhline(-5, color="gray", ls=":", label="гроші на рахунку (−5 %)")
a1.axhline(SUMMARY["bh_annual"], color="k", ls="--", lw=0.9, label="«купив і тримай»")
a1.set_xlabel("відносний рівень шуму λ"); a1.set_ylabel("річний приріст, %"); a1.legend(fontsize=8)
a1.set_title("а) вплив дисперсії шуму")
d30 = g_[(g_.g == 30) & (g_.mu == 0)]
fl = nz[nz.kind == "flip"]
a2.plot(d30.sign, d30.annual, "o-", color="tab:orange", label="гаусів шум, g = 30 %")
a2.plot(np.r_[100, fl.sign], np.r_[d30.annual.iloc[0], fl.annual], "s--", color="tab:purple", label="точна величина, випадковий знак")
sg = nz[nz.kind == "sign"]
a2.plot(sg.sign, sg.annual, "*", ms=14, color="tab:red", label="правильний знак, випадкова величина")
a2.axhline(-5, color="gray", ls=":")
a2.set_xlabel("частка вгаданих знаків, %"); a2.set_ylabel("річний приріст, %"); a2.legend(fontsize=8)
a2.set_title("б) роль знака прогнозу")
fig.tight_layout(); fig.savefig("figs/fig6_noise.png"); plt.show()

def pick(**kw):
    d = nz
    for k_, v in kw.items():
        d = d[np.isclose(d[k_], v)] if isinstance(v, float) else d[d[k_] == v]
    return d.iloc[0]

for lam_, tag in ((0, "0"), (0.5, "05"), (1.0, "1"), (2.0, "2"), (3.0, "3")):
    for g in (10, 30, 100):
        SUMMARY[f"n_g{g}_{tag}"] = pick(kind="gauss", g=g, lam=float(lam_), mu=0.0)["annual"]
    SUMMARY[f"n_sign_{tag}"] = pick(kind="gauss", g=30, lam=float(lam_), mu=0.0)["sign"]
    SUMMARY[f"n_corr_{tag}"] = pick(kind="gauss", g=30, lam=float(lam_), mu=0.0)["corr"]
    SUMMARY[f"n_inv_{tag}"] = pick(kind="gauss", g=30, lam=float(lam_), mu=0.0)["inv"]
for mu_, tag in ((-0.5, "m05"), (-0.25, "m025"), (0.0, "0"), (0.25, "p025"), (0.5, "p05")):
    SUMMARY[f"b_{tag}"] = pick(kind="gauss", g=30, lam=0.5, mu=mu_)["annual"]
SUMMARY["b_perfect"] = pick(kind="gauss", g=30, lam=0.0, mu=0.0)["annual"]
SUMMARY["sign_only"] = nz[nz.kind == "sign"]["annual"].iloc[0]
for fl_ in (0.1, 0.2, 0.3, 0.4, 0.5):
    SUMMARY[f"f_{str(fl_).replace('.', '')}"] = pick(kind="flip", flip=fl_)["annual"]

## Задача 4. Стандартні моделі прогнозування
Навчальна ділянка — перші 60 % даних, тестова — решта. Моделі: середнє (наївна), AR(p) з
рекурсивним МНК, ARMA(p,q) для дохідностей, ARIMA(p,1,q) для логарифма ціни, метод Хольта
з загасаючим трендом. Прогноз будується на 5 кроків уперед у кожний момент тестової ділянки.

In [ ]:
T_TRAIN = int(round(0.6 * K))
LP = np.log(P)
SUMMARY.update(t0=T_TRAIN, test_start=str(px.index[T_TRAIN].date()), test_end=str(px.index[-1].date()), KT=K - T_TRAIN)
st_rows = []
for i, c in enumerate(names):
    st_rows.append(dict(asset=c, adf_price=adfuller(LP[:T_TRAIN + 1, i], autolag="AIC")[1],
                        adf_ret=adfuller(XI[:T_TRAIN, i], autolag="AIC")[1],
                        lb10=acorr_ljungbox(XI[:T_TRAIN, i], lags=[10])["lb_pvalue"].iloc[0],
                        jb=jarque_bera(XI[:T_TRAIN, i]).pvalue, kurt=kurtosis(XI[:T_TRAIN, i])))
tab_stats = pd.DataFrame(st_rows)
tab_stats.to_csv("results/tab_stats.csv", index=False)
display(tab_stats.round(4))
SUMMARY.update(adf_price_min=tab_stats.adf_price.min(), adf_ret_max=tab_stats.adf_ret.max(),
               lb_sig=int((tab_stats.lb10 < 0.05).sum()), lb_sig_assets=", ".join(tab_stats.asset[tab_stats.lb10 < 0.05]),
               jb_sig=int((tab_stats.jb < 0.05).sum()), kurt_max=tab_stats["kurt"].max(),
               kurt_max_asset=tab_stats.asset[tab_stats["kurt"].idxmax()])

In [ ]:
def ar_design(x, p):
    Xr = np.column_stack([np.ones(len(x) - p)] + [x[p - j - 1: len(x) - j - 1] for j in range(p)])
    return Xr, x[p:]

def ar_order_aic(x, pmax=5):
    best = (np.inf, 1)
    for p in range(1, pmax + 1):
        Xr, yv = ar_design(x[pmax - p:], p)
        th = np.linalg.lstsq(Xr, yv, rcond=None)[0]
        best = min(best, (len(yv) * np.log(np.mean((yv - Xr @ th) ** 2)) + 2 * (p + 1), p))
    return best[1]

class RLS:
    """Рекурсивний МНК з коефіцієнтом забування lam."""

    def __init__(self, theta, P, lam=0.99):
        self.th, self.P, self.lam = theta.copy(), P.copy(), lam

    def update(self, phi, y):
        Pphi = self.P @ phi
        k = Pphi / (self.lam + phi @ Pphi)
        self.th = self.th + k * (y - phi @ self.th)
        self.P = (self.P - np.outer(k, Pphi)) / self.lam

def forecast_mean(t0):
    F = np.zeros((K, NP, n))
    for k in range(t0, K):
        F[k] = XI[:k].mean(0)
    return F

def forecast_ar_rls(t0, pmax=5, lam=0.99):
    F = np.zeros((K, NP, n)); orders = []
    for i in range(n):
        x = XI[:, i]
        p = ar_order_aic(x[:t0], pmax); orders.append(p)
        Xr, yv = ar_design(x[:t0], p)
        est = RLS(np.linalg.lstsq(Xr, yv, rcond=None)[0], np.linalg.inv(Xr.T @ Xr), lam)
        for k in range(t0, K):
            if k > t0:
                est.update(np.r_[1.0, x[k - 1 - p:k - 1][::-1]], x[k - 1])
            hist = list(x[:k])
            for h in range(NP):
                yh = float(np.r_[1.0, np.array(hist[-p:])[::-1]] @ est.th)
                F[k, h, i] = yh; hist.append(yh)
    return F, orders

def select_arma(x, d, trend):
    best = (np.inf, None)
    for p, q in ((0, 1), (1, 0), (1, 1), (2, 0), (0, 2), (2, 1), (1, 2), (2, 2)):
        try:
            aic = ARIMA(x, order=(p, d, q), trend=trend).fit().aic
            best = min(best, (aic, (p, q)))
        except Exception:
            pass
    return best[1]

def forecast_arima_one(series, t0, d, trend, refit=21):
    """d=0: ARMA для дохідностей; d=1: ARIMA для log-ціни (прогноз переводимо в дохідності)."""
    Kx = len(series) - (1 if d == 1 else 0)
    F = np.zeros((Kx, NP))
    p, q = select_arma(series[:t0 + d], d, trend)
    params = None
    for k in range(t0, Kx):
        end = k + d
        if params is None or (k - t0) % refit == 0:
            params = ARIMA(series[:end], order=(p, d, q), trend=trend).fit().params
        fc = np.asarray(ARIMA(series[:end], order=(p, d, q), trend=trend).filter(params).forecast(NP))
        F[k] = fc if d == 0 else np.exp(np.diff(np.r_[series[end - 1], fc])) - 1
    return F, (p, q)

def forecast_holt_one(logp, t0, refit=21):
    Kx = len(logp) - 1
    F = np.zeros((Kx, NP)); prm = None
    for k in range(t0, Kx):
        m = ExponentialSmoothing(logp[:k + 1], trend="add", damped_trend=True)
        if prm is None or (k - t0) % refit == 0:
            r = m.fit(optimized=True)
            prm = {kk: r.params[kk] for kk in ("smoothing_level", "smoothing_trend", "damping_trend")}
        else:
            r = m.fit(optimized=False, **prm)
        F[k] = np.exp(np.diff(np.r_[logp[k], np.asarray(r.forecast(NP))])) - 1
    return F

def std_one_asset(i):
    Fa, oa = forecast_arima_one(XI[:, i], T_TRAIN, 0, "c")
    Fi, oi = forecast_arima_one(LP[:, i], T_TRAIN, 1, "t")
    return i, Fa, oa, Fi, oi, forecast_holt_one(LP[:, i], T_TRAIN)

t = time.time()
FC = {"mean": forecast_mean(T_TRAIN)}
FC["ar_rls"], ar_orders = forecast_ar_rls(T_TRAIN)
with Pool(NPROC) as pool:
    outs = pool.map(std_one_asset, range(n), chunksize=1)
for key in ("arma", "arima", "holt"):
    FC[key] = np.zeros((K, NP, n))
orders = {"AR": {}, "ARMA": {}, "ARIMA": {}}
for i, Fa, oa, Fi, oi, Fh in outs:
    FC["arma"][:, :, i], FC["arima"][:, :, i], FC["holt"][:, :, i] = Fa, Fi, Fh
    orders["AR"][names[i]] = ar_orders[i]; orders["ARMA"][names[i]] = oa; orders["ARIMA"][names[i]] = oi
np.savez_compressed("results/forecasts_std.npz", **FC)
json.dump(orders, open("results/orders.json", "w"), indent=1)
print("час прогнозування:", round(time.time() - t), "c")
print(orders)

In [ ]:
def accuracy(F, h=0):
    ks = np.arange(T_TRAIN, K - h)
    yp, yt = F[ks, h, :].ravel(), XI[ks + h, :].ravel()
    return dict(R2=1 - np.sum((yt - yp) ** 2) / np.sum((yt - yt.mean()) ** 2), RMSE=np.sqrt(np.mean((yt - yp) ** 2)) * 100,
                MAE=np.mean(np.abs(yt - yp)) * 100,
                sign=(np.mean(np.sign(yp) == np.sign(yt)) + 0.5 * np.mean(yp == 0)) * 100,
                corr=np.corrcoef(yp, yt)[0, 1], pos=np.mean(yp > 0) * 100, mag=np.mean(np.abs(yp)) * 100)

acc_std = pd.DataFrame([dict(model=m_, h=h + 1, **accuracy(FC[m_], h)) for m_ in FC for h in (0, 4)])
acc_std.to_csv("results/tab_acc_std.csv", index=False)
display(acc_std.round(4))

fig, ax = plt.subplots(figsize=(11, 4))
dts = px.index[T_TRAIN + 1:]
iMU = names.index("MU")
ax.bar(dts, XI[T_TRAIN:, iMU] * 100, width=1.0, color="lightgray", label="фактична дохідність")
for key, lab in (("ar_rls", "AR + РМНК"), ("arma", "ARMA"), ("arima", "ARIMA"), ("holt", "Хольт")):
    ax.plot(dts, FC[key][T_TRAIN:, 0, iMU] * 100, lw=1.2, label=f"прогноз {lab}")
ax.set_ylabel("%"); ax.legend(ncol=5, fontsize=8); ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))
fig.tight_layout(); fig.savefig("figs/fig7_forecast_mu.png"); plt.show()

In [ ]:
XT = XI[T_TRAIN:]; KT = len(XT); PT = P[T_TRAIN:]

def run_test(job):
    """Керування на тестовій ділянці за прогнозом name; ref — спосіб задання траєкторії."""
    name, g_ann, ref, F = job
    fc = (lambda k: XT[k:k + NP]) if name == "perfect" else (lambda k: F[T_TRAIN + k])
    s = simulate(model, MPC(model, NP, 1.0, RHO, (1 + g_ann) ** (1 / 252) - 1, solver="quadprog", ref=ref), XT, fc, X0)
    return dict(model=name, g=g_ann * 100, ref=ref, **metrics(s["y"]), fees=s["fees"],
                inv=(s["X"][:, :n].sum(1) / s["y"]).mean() * 100, y=s["y"])

for sd in range(3):
    rng = np.random.default_rng(100 + sd)
    FC[f"random{sd}"] = rng.standard_normal((K, NP, n)) * S_I
jobs = [(m_, g, "relative", FC.get(m_)) for g in (0.1, 0.3) for m_ in ["perfect", "mean", "ar_rls", "arma", "arima", "holt",
                                                                         "random0", "random1", "random2"]]
with Pool(NPROC) as pool:
    res_std = pool.map(run_test, jobs, chunksize=1)
ew_t = (1e5 / (n * 1.0005) * PT / PT[0]).sum(1)
cash_t = 1e5 * (1 - r_step) ** np.arange(KT + 1)
bench = [dict(model="buy_hold", g=np.nan, ref="", **metrics(ew_t), fees=0, inv=100, y=ew_t),
         dict(model="cash", g=np.nan, ref="", **metrics(cash_t), fees=0, inv=0, y=cash_t)]
tab_mpc_std = pd.DataFrame([{k: v for k, v in r.items() if k != "y"} for r in res_std + bench])
tab_mpc_std.to_csv("results/tab_mpc_std.csv", index=False)
display(tab_mpc_std.round(2))

fig, ax = plt.subplots(figsize=(11, 4.8))
lab = {"perfect": "ідеальний прогноз", "mean": "середнє", "ar_rls": "AR + РМНК", "arma": "ARMA", "arima": "ARIMA", "holt": "Хольт"}
for r in res_std:
    if r["g"] == 30 and r["model"] in lab:
        ax.plot(px.index[T_TRAIN:], r["y"], lw=2 if r["model"] == "perfect" else 1.2, label=lab[r["model"]])
ax.plot(px.index[T_TRAIN:], ew_t, "k--", lw=1, label="«купив і тримай»")
ax.plot(px.index[T_TRAIN:], cash_t, color="gray", ls=":", label="гроші на рахунку")
ax.set_ylabel("капітал, $"); ax.legend(ncol=3, fontsize=8); ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))
fig.tight_layout(); fig.savefig("figs/fig8_std_capital.png"); plt.show()

In [ ]:
# Порівняння способів задання траєкторії: від поточного капіталу (relative) і від початкового (absolute)
jobs = []
for lam in (0.5, 1.0, 2.0, 3.0):
    rng = np.random.default_rng(0)
    E = rng.standard_normal((K, NP, n))
    FC[f"noise{lam}"] = [XI[k:k + NP] + lam * E[k][:len(XI[k:k + NP])] * S_I for k in range(K)]
for ref in ("relative", "absolute"):
    jobs += [(f"noise{lam}", 0.3, ref, FC[f"noise{lam}"]) for lam in (0.5, 1.0, 2.0, 3.0)]
    jobs += [(m_, 0.3, ref, FC[m_]) for m_ in ("mean", "ar_rls", "arima", "holt")]

def run_ref(job):
    name, g_ann, ref, F = job
    if name.startswith("noise"):
        s = simulate(model, MPC(model, NP, 1.0, RHO, (1 + g_ann) ** (1 / 252) - 1, solver="quadprog", ref=ref), XI, lambda k: F[k], X0)
        return dict(model=name, ref=ref, **metrics(s["y"]), inv=(s["X"][:, :n].sum(1) / s["y"]).mean() * 100)
    r = run_test(job)
    return {k: v for k, v in r.items() if k in ("model", "ref", "final", "total", "annual", "maxdd", "inv")}

with Pool(NPROC) as pool:
    tab_ref = pd.DataFrame(pool.map(run_ref, jobs, chunksize=1))
tab_ref.to_csv("results/tab_ref.csv", index=False)
display(tab_ref.pivot(index="model", columns="ref", values=["annual", "maxdd", "inv"]).round(1))

## Задача 5. Нейронні мережі та ансамбль
Глобальна модель для всіх 22 активів: ознаки — 5 лагів дохідності, ковзні середні, відношення
волатильностей, відхилення від MA20, середня дохідність усіх 22 активів за кожен із двох останніх днів (стан ринку), лаг BTC. Ціль —
дохідність на h кроків уперед, нормована на волатильність. Окрема модель для кожного h = 1..5.
Перенавчання раз на 21 торговий день на розширюваному вікні.

In [ ]:
W = 20
iBTC = names.index("BTC")

def features(k):
    R = XI[k - W:k]; sig = R.std(0) + 1e-12
    mk = R.mean(1); smk = mk.std() + 1e-12
    F = np.column_stack([R[-j] / sig for j in range(1, 6)] +
                        [R[-5:].mean(0) / sig, R.mean(0) / sig, R[-5:].std(0) / sig,
                         (LP[k] - LP[k - W + 1:k + 1].mean(0)) / (sig * np.sqrt(W)),
                         np.full(n, mk[-1] / smk), np.full(n, mk[-2] / smk), np.full(n, XI[k - 1, iBTC] / sig[iBTC])])
    return F, sig

FE = {k: features(k) for k in range(W, K)}

def ml_block(k_now):
    """Навчання на даних, відомих до k_now, і прогноз для наступних 21 кроків."""
    ks = list(range(k_now, min(k_now + 21, K)))
    out = {m_: np.zeros((len(ks), NP, n)) for m_ in ("mlp", "hgb", "ridge", "ens")}
    for h in range(NP):
        X_, y_ = [], []
        for k in range(W, k_now - h):
            f_, s_ = FE[k]; X_.append(f_); y_.append(XI[k + h] / s_)
        X_, y_ = np.vstack(X_), np.concatenate(y_)
        sc = StandardScaler().fit(X_); Xs = sc.transform(X_)
        mlps = [MLPRegressor(hidden_layer_sizes=(32, 16), alpha=1e-2, learning_rate_init=1e-3, max_iter=400,
                             early_stopping=True, validation_fraction=0.15, n_iter_no_change=20, random_state=sd).fit(Xs, y_)
                for sd in range(5)]
        hgb = HistGradientBoostingRegressor(max_depth=3, learning_rate=0.05, max_iter=200, min_samples_leaf=50,
                                            l2_regularization=1.0, random_state=0).fit(Xs, y_)
        rg = Ridge(alpha=10.0).fit(Xs, y_)
        for j, k in enumerate(ks):
            f_, s_ = FE[k]; fs = sc.transform(f_)
            zm = np.mean([mm.predict(fs) for mm in mlps], axis=0); zh = hgb.predict(fs); zr = rg.predict(fs)
            out["mlp"][j, h], out["hgb"][j, h], out["ridge"][j, h] = zm * s_, zh * s_, zr * s_
            out["ens"][j, h] = (zm + zh + zr) / 3 * s_
    return ks, out

t = time.time()
FML = {m_: np.zeros((K, NP, n)) for m_ in ("mlp", "hgb", "ridge", "ens")}
with Pool(NPROC) as pool:
    for ks, out in pool.map(ml_block, range(T_TRAIN, K, 21), chunksize=1):
        for m_ in FML:
            FML[m_][ks] = out[m_]
np.savez_compressed("results/forecasts_ml.npz", **FML)
print("час навчання:", round(time.time() - t), "c")
acc_ml = pd.DataFrame([dict(model=m_, h=h + 1, **accuracy(FML[m_], h)) for m_ in FML for h in (0, 4)])
acc_ml.to_csv("results/tab_acc_ml.csv", index=False)
display(acc_ml.round(4))

In [ ]:
jobs = [(m_, g, "relative", FML[m_]) for g in (0.1, 0.3) for m_ in FML]
with Pool(NPROC) as pool:
    res_ml = pool.map(run_test, jobs, chunksize=1)
tab_mpc_ml = pd.DataFrame([{k: v for k, v in r.items() if k != "y"} for r in res_ml])
tab_mpc_ml.to_csv("results/tab_mpc_ml.csv", index=False)
display(tab_mpc_ml.round(2))

fig, ax = plt.subplots(figsize=(11, 4.8))
labm = {"mlp": "MLP (ансамбль 5 мереж)", "hgb": "градієнтний бустинг", "ens": "ансамбль MLP+бустинг+Ridge"}
for r in res_std:
    if r["g"] == 30 and r["model"] in ("perfect", "arima"):
        ax.plot(px.index[T_TRAIN:], r["y"], lw=2 if r["model"] == "perfect" else 1.2, label=lab[r["model"]])
for r in res_ml:
    if r["g"] == 30 and r["model"] in labm:
        ax.plot(px.index[T_TRAIN:], r["y"], lw=1.2, label=labm[r["model"]])
ax.plot(px.index[T_TRAIN:], ew_t, "k--", lw=1, label="«купив і тримай»")
ax.plot(px.index[T_TRAIN:], cash_t, color="gray", ls=":", label="гроші на рахунку")
ax.set_ylabel("капітал, $"); ax.legend(ncol=3, fontsize=8); ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))
fig.tight_layout(); fig.savefig("figs/fig9_ml_capital.png"); plt.show()

## Задача 6. Модель авторегресії курсу BTC
Вихід — денна лог-дохідність BTC, %. Спочатку чиста AR(p) (вхід — шум), потім ARX з вимірюваним
входом — дисбалансом агресивних купівель і продажів $u=100(2V_b/V-1)$. Передаточна функція,
нулі/полюси, канонічна керована форма, ранги матриць керованості та спостережуваності.

In [ ]:
bd = btc_d.copy()
bd.index = bd.index.tz_localize(None) if bd.index.tz is not None else bd.index
bd = bd.loc[:"2026-10-16"]
dfb = pd.DataFrame({"y": 100 * np.log(bd["close"]).diff(), "u": 100 * (2 * bd["taker_buy_quote"] / bd["quote_volume"] - 1),
                    "qv": bd["quote_volume"], "lp": 100 * np.log(bd["close"])}).dropna()
est, val = dfb.loc[:"2025-12-31"].copy(), dfb.loc["2026-01-01":].copy()
U0 = est["u"].mean(); est["u"] -= U0; val["u"] -= U0

def design(data, p, m, yname="y", with_u=True):
    cols = [np.ones(len(data))] + [data[yname].shift(l).values for l in range(1, p + 1)]
    if with_u:
        cols += [data["u"].shift(l).values for l in range(0, m + 1)]
    return np.column_stack(cols), data[yname].values

def ols(data, p, m, yname="y", with_u=True, start=0):
    X, Y = design(data, p, m, yname, with_u)
    ok = ~np.isnan(X).any(1); ok[:start] = False
    th = np.linalg.lstsq(X[ok], Y[ok], rcond=None)[0]
    return th, Y[ok] - X[ok] @ th, Y[ok]

def ic(e, k):
    N, s2 = len(e), np.mean(e ** 2)
    return N * np.log(s2) + 2 * k, N * np.log(s2) + k * np.log(N)

def r2(y, e):
    return 1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2)

def predict(th, data, p, m, with_u=True):
    X, Y = design(data, p, m, with_u=with_u); ok = ~np.isnan(X).any(1)
    return X[ok] @ th, Y[ok], data.index[ok]

tab_ar = pd.DataFrame([(p,) + ic(ols(est, p, 0, with_u=False, start=10)[1], p + 1) for p in range(1, 11)], columns=["p", "AIC", "BIC"])
p_ar = int(tab_ar.loc[tab_ar.AIC.idxmin(), "p"]); p_ar_bic = int(tab_ar.loc[tab_ar.BIC.idxmin(), "p"])
th_ar, e_ar, y_ar = ols(est, p_ar, 0, with_u=False)
yh, yv, _ = predict(th_ar, val, p_ar, 0, with_u=False)
den_ar = np.r_[1.0, -th_ar[1:]]
SUMMARY.update(btc_N=len(dfb), btc_Nest=len(est), btc_Nval=len(val), ar_p=p_ar, ar_p_bic=p_ar_bic, ar_c=th_ar[0],
               ar_a1=th_ar[1], ar_poles=np.abs(np.roots(den_ar)).max(), ar_r2=r2(y_ar, e_ar), ar_r2v=r2(yv, yv - yh),
               ar_lb=acorr_ljungbox(e_ar, lags=[10])["lb_pvalue"].iloc[0])
tab_arp = pd.DataFrame([(p,) + ic(ols(est, p, 0, yname="lp", with_u=False, start=5)[1], p + 1) for p in range(1, 6)], columns=["p", "AIC", "BIC"])
p_lp = int(tab_arp.loc[tab_arp.BIC.idxmin(), "p"])
th_lp = ols(est, p_lp, 0, yname="lp", with_u=False)[0]
SUMMARY.update(arp_p=p_lp, arp_a1=th_lp[1], arp_pole=np.abs(np.roots(np.r_[1.0, -th_lp[1:]])).max())

grid = []
for p in range(0, 6):
    for m in range(0, 4):
        grid.append((p, m) + ic(ols(est, p, m, start=6)[1], 2 + p + m))
grid = pd.DataFrame(grid, columns=["p", "m", "AIC", "BIC"])
best_bic, best_aic = grid.loc[grid.BIC.idxmin()], grid.loc[grid.AIC.idxmin()]
p_x, m_x = int(best_bic.p), int(best_bic.m)
th_x, e_x, y_x = ols(est, p_x, m_x)
a_x, b_x = th_x[1:1 + p_x], th_x[1 + p_x:]
Nn = max(p_x, m_x)
den_x = np.r_[1.0, -a_x, np.zeros(Nn - p_x)]
num_x = np.r_[b_x, np.zeros(Nn + 1 - len(b_x))]
zeros_x, poles_x = np.roots(num_x), np.roots(den_x)
A_s, B_s, C_s, D_s = signal.tf2ss(num_x, den_x)
nA = A_s.shape[0]
Co = np.hstack([np.linalg.matrix_power(A_s, i) @ B_s for i in range(nA)])
Ob = np.vstack([C_s @ np.linalg.matrix_power(A_s, i) for i in range(nA)])
yh_x, yv_x, idx_v = predict(th_x, val, p_x, m_x)
SUMMARY.update(u0=U0, arx_p=p_x, arx_m=m_x, arx_p_aic=int(best_aic.p), arx_m_aic=int(best_aic.m), arx_c=th_x[0],
               arx_b0=b_x[0], arx_b1=b_x[1] if len(b_x) > 1 else 0.0, arx_zero=float(np.real(zeros_x[0])) if len(zeros_x) else np.nan,
               arx_pole=float(np.abs(poles_x).max()), arx_gain=float(b_x.sum() / (1 - a_x.sum())),
               arx_r2=r2(y_x, e_x), arx_r2v=r2(yv_x, yv_x - yh_x), arx_rank_c=int(np.linalg.matrix_rank(Co)),
               arx_rank_o=int(np.linalg.matrix_rank(Ob)), arx_n=nA, arx_A=A_s.tolist(), arx_B=B_s.tolist(), arx_C=C_s.tolist(),
               arx_D=D_s.tolist(), arx_du=1 / b_x[0], arx_qv=est["qv"].median() / 1e9,
               arx_money=1 / b_x[0] / 100 * est["qv"].median() / 1e6, arx_kurt=kurtosis(e_x),
               arx_jb=jarque_bera(e_x).statistic, arx_lb=acorr_ljungbox(e_x, lags=[10])["lb_pvalue"].iloc[0])
sig = np.sqrt(pd.Series(e_x).pow(2).ewm(alpha=0.06).mean().shift(1)).values
SUMMARY["norm_kurt"] = kurtosis((e_x / sig)[30:])
grid.to_csv("results/tab_arx_grid.csv", index=False); tab_ar.to_csv("results/tab_ar_btc.csv", index=False)
print({k: SUMMARY[k] for k in SUMMARY if k.startswith(("ar", "arx", "btc_N", "norm", "u0"))})

fig = plt.figure(figsize=(12, 7.5))
ax1 = fig.add_subplot(2, 1, 1)
nshow = 120
ax1.plot(idx_v[:nshow], yv_x[:nshow], color="lightgray", lw=1.5, label="фактична дохідність BTC")
ax1.plot(idx_v[:nshow], yh_x[:nshow], color="tab:blue", lw=1.0, label="модель ARX за дисбалансом потоку заявок")
ax1.set_ylabel("%"); ax1.legend(); ax1.set_title("а) перевірна вибірка (2026 рік), перші 120 днів")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
ax2 = fig.add_subplot(2, 2, 3)
th_c = np.linspace(0, 2 * np.pi, 400)
ax2.plot(np.cos(th_c), np.sin(th_c), "k-", lw=0.8)
ax2.plot(np.real(poles_x), np.imag(poles_x), "rx", ms=10, mew=2, label="полюс")
ax2.plot(np.real(zeros_x), np.imag(zeros_x), "bo", ms=8, mfc="none", mew=2, label="нуль")
ax2.plot([SUMMARY["arp_pole"]], [0], "m+", ms=12, mew=2, label="полюс AR для log-ціни")
ax2.set_aspect("equal"); ax2.set_xlim(-1.3, 1.3); ax2.set_ylim(-1.3, 1.3); ax2.legend(fontsize=8, loc="lower left")
ax2.set_title("б) нулі та полюси $W_u(z)$")
ax3 = fig.add_subplot(2, 2, 4)
_, st_y = signal.dstep((num_x, den_x, 1), n=15)
st_y = np.squeeze(st_y)
ax3.step(range(15), st_y, where="post", label="дохідність, %")
ax3.step(range(15), np.cumsum(st_y), where="post", label="log-ціна, % (інтегратор)")
ax3.set_xlabel("дні"); ax3.legend(fontsize=8); ax3.set_title("в) перехідна характеристика на $\\Delta u$ = 1 в. п.")
fig.tight_layout(); fig.savefig("figs/fig11_btc.png"); plt.show()